# Confusion Matrix
* True Positives: TPs are the cases when the actual class of data point was 1 and
the predicted is also 1.
* True Negatives: TNs are the cases when the actual class of the data point was 0
and the predicted is also 0.
*  False Positives: FPs are the cases when the actual class of data point was 0 and
the predicted is also 1.
* False Negatives: FNs are the cases when the actual class of the data point was 1
and the predicted is also 0.

# Classification Report

* **Precision** : It may be defined as how many of the returned
documents are correct.
* **Recall or Sensitivity** : It may be defined as how many of the positives do the model return.
* **f1-score** : It gives a combined idea about Precision and Recall metrics. It is maximum when Precision is equal to Recall.

# Class Imbalance Problem

In [28]:
import pandas as pd
from sklearn.datasets import make_classification

# Generate synthetic imbalanced binary classification dataset
X, y = make_classification(
    n_samples=1000,           # Total number of rows
    n_features=5,             # Total features
    n_informative=3,         # Useful features for prediction
    n_redundant=1,           # Correlated features
    weights=[0.90, 0.10],    # Imbalance ratio: 90% Class 0 (Majority), 10% Class 1 (Minority)
    flip_y=0.01,             # Noise level (1% labels flipped)
    random_state=42
)

# Convert into a Pandas DataFrame
feature_cols = [f"feature_{i+1}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_cols)
df["target"] = y

# Inspect class distribution
print("Class breakdown:")
print(df["target"].value_counts(normalize=True))    

df.head()

X = df[["feature_1","feature_2","feature_3","feature_4","feature_5"]]

y = df.target

X_train , X_test , y_train, y_test = train_test_split(X,y,test_size=0.40,random_state=42)

from sklearn.linear_model import LogisticRegression

LR = LogisticRegression(class_weight="balanced")

LR.fit(X_train,y_train)

y_pred = LR.predict(X_test)

from sklearn.metrics import classification_report , confusion_matrix

print(confusion_matrix(y_test,y_pred))

print(50*"*")

print(classification_report(y_test,y_pred))

Class breakdown:
target
0    0.898
1    0.102
Name: proportion, dtype: float64
[[321  37]
 [  4  38]]
**************************************************
              precision    recall  f1-score   support

           0       0.99      0.90      0.94       358
           1       0.51      0.90      0.65        42

    accuracy                           0.90       400
   macro avg       0.75      0.90      0.79       400
weighted avg       0.94      0.90      0.91       400



# How To Solve The Problem ? 

Balancing the classes’ acts as a solution to imbalanced classes. The main objective of
balancing the classes is to either increase the frequency of the minority class or decrease
the frequency of the majority class. Following are the approaches to solve the issue of
imbalances classes:

# Re-Sampling
Re-sampling is a series of methods used to reconstruct the sample data sets - both training
sets and testing sets. Re-sampling is done to improve the accuracy of model. Following
are some re-sampling techniques:

## Random Under-Sampling
* This technique aims to balance class distribution by
randomly eliminating majority class examples. This is done until the majority and
minority class instances are balanced out.

In [29]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy="auto",random_state=42)

X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

LR.fit(X_train_resampled,y_train_resampled)

y_pred = LR.predict(X_test)

from sklearn.metrics import classification_report , confusion_matrix

print(confusion_matrix(y_test,y_pred))

print(50*"*")

print(classification_report(y_test,y_pred))


[[316  42]
 [  4  38]]
**************************************************
              precision    recall  f1-score   support

           0       0.99      0.88      0.93       358
           1       0.47      0.90      0.62        42

    accuracy                           0.89       400
   macro avg       0.73      0.89      0.78       400
weighted avg       0.93      0.89      0.90       400



In [30]:
print("Original train set class balance:\n", pd.Series(y_train).value_counts())
print("\nResampled train set class balance:\n", pd.Series(y_train_resampled).value_counts())

Original train set class balance:
 target
0    540
1     60
Name: count, dtype: int64

Resampled train set class balance:
 target
0    60
1    60
Name: count, dtype: int64


## Random Over-Sampling
*  This technique aims to balance class distribution by
increasing the number of instances in the minority class by replicating them.

In [31]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(sampling_strategy="auto",random_state=42)

X_train_resampled, y_train_resampled = ros.fit_resample(X_train, y_train)

LR.fit(X_train_resampled,y_train_resampled)

y_pred = LR.predict(X_test)

from sklearn.metrics import classification_report , confusion_matrix

print(confusion_matrix(y_test,y_pred))

print(50*"*")

print(classification_report(y_test,y_pred))




[[320  38]
 [  4  38]]
**************************************************
              precision    recall  f1-score   support

           0       0.99      0.89      0.94       358
           1       0.50      0.90      0.64        42

    accuracy                           0.90       400
   macro avg       0.74      0.90      0.79       400
weighted avg       0.94      0.90      0.91       400



In [32]:
print("Original train set class balance:\n", pd.Series(y_train).value_counts())
print("\nResampled train set class balance:\n", pd.Series(y_train_resampled).value_counts())

Original train set class balance:
 target
0    540
1     60
Name: count, dtype: int64

Resampled train set class balance:
 target
0    540
1    540
Name: count, dtype: int64


__The main advantage of this method is that there would be no loss of useful information.
But on the other hand, it has the increased chances of over-fitting because it replicates
the minority class events.__

## When Should You Actually Use Resampling?
- Resampling techniques like ROS/RUS are not guaranteed to improve performance across all datasets or metrics. They are meant to solve specific scenarios:

- Extreme Imbalance: Ratios like 99:1 or 99.9:0.1 (e.g., credit card fraud), where the classifier completely ignores the minority class and gets 0% recall.

- High Overlap / Heavy Noise: When minority samples are buried deep inside majority clusters.

- Algorithms Sensitive to Prior Probabilities: Standard Decision Trees or SVMs that don't calibrate probabilities well.